# UC1 — PG with all 4 sidecars + multi-version + dual-search

**Persona:** platform builder ingesting versioned vector features.

**Goal:** stand up a collection that:
1. stores items on PostgreSQL with all four sidecars (`geometries`,
   `attributes`, `item_metadata`, `stac_metadata`) — full default surface;
2. enables 2D geometry statistics (area, length, centroid) as columns,
   indexed and exposed on the feature type;
3. uses an `ItemsWritePolicy` keyed on `properties.code` so re-ingesting
   the same `code` from a different asset creates a **new version** rather
   than overwriting (`asset_id` tracking is enabled);
4. routes WRITE to PG (fatal) + ES public (async) and SEARCH/READ to
   ES public with `geometry_simplified` + PG with `geometry_exact`,
   so callers can pick approximate-fast vs exact-slow at query time.

**Critical:** `enable_validity=true` is required for `on_conflict=new_version`
to actually create version rows; without it the policy silently degrades to
`update`.  We set it explicitly.

**Critical:** sidecars are immutable post-creation — the four-sidecar set
below MUST be applied before any feature is ingested.


In [ ]:
import json
import os
import time
import uuid

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
TOKEN = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)
RUN_ID = os.environ.get("RUN_ID", uuid.uuid4().hex[:8])
CATALOG_ID = os.environ.get("CATALOG_ID", f"cf_uc1_{RUN_ID}")
COLLECTION_ID = os.environ.get("COLLECTION_ID", f"col_{RUN_ID}")

IS_LOCAL = "localhost" in BASE_URL or "127.0.0.1" in BASE_URL

headers = {"Content-Type": "application/json"}
if TOKEN:
    headers["Authorization"] = f"Bearer {TOKEN}"

client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=120.0)

print(f"BASE_URL      : {BASE_URL}")
print(f"CATALOG_ID    : {CATALOG_ID}")
print(f"COLLECTION_ID : {COLLECTION_ID}")
print(f"AUTH          : {'token set' if TOKEN else 'anonymous'}")
if not TOKEN:
    print("\nWARNING: no Bearer token set — config writes will 401.")
    print("Set DYNASTORE_TOKEN before running write cells.")


In [ ]:
# Create catalog (idempotent: 409 = already exists)
catalog_payload = {
    "id": CATALOG_ID,
    "type": "Catalog",
    "title": f"Cycle F UC walkthrough {RUN_ID}",
    "description": "Ephemeral catalog for cycle_f_use_cases notebook.",
    "stac_version": "1.0.0",
}
r = client.post("/stac/catalogs", content=json.dumps(catalog_payload))
if r.status_code in (200, 201):
    print(f"Catalog '{CATALOG_ID}' created.")
elif r.status_code == 409:
    print(f"Catalog '{CATALOG_ID}' already exists.")
else:
    raise RuntimeError(f"Catalog create failed: {r.status_code}: {r.text}")

# Create collection (idempotent)
collection_payload = {
    "id": COLLECTION_ID,
    "type": "Collection",
    "stac_version": "1.0.0",
    "title": f"UC collection {RUN_ID}",
    "description": "Walkthrough collection — defaults inherited then PATCHed.",
    "license": "proprietary",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]},
    },
    "links": [],
}
r = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections",
    content=json.dumps(collection_payload),
)
if r.status_code in (200, 201):
    print(f"Collection '{COLLECTION_ID}' created.")
elif r.status_code == 409:
    print(f"Collection '{COLLECTION_ID}' already exists.")
else:
    raise RuntimeError(f"Collection create failed: {r.status_code}: {r.text}")


In [ ]:
# Helper — show explicit-vs-effective config delta for a plugin
def show_config_delta(plugin_id: str, level: str = "collection") -> None:
    if level == "collection":
        base = f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}"
    elif level == "catalog":
        base = f"/configs/catalogs/{CATALOG_ID}"
    else:
        raise ValueError(level)
    rx = client.get(f"{base}/plugins/{plugin_id}")
    re_ = client.get(f"{base}/plugins/{plugin_id}/effective")
    print(f"\n=== {plugin_id} @ {level} ===")
    print("EXPLICIT:")
    if rx.status_code == 200:
        print(json.dumps(rx.json(), indent=2)[:600])
    else:
        print(f"  ({rx.status_code} — none stored, every field inherited)")
    if re_.status_code != 200:
        print(f"  effective unavailable ({re_.status_code})")
        return
    eff = re_.json()
    resolved = eff.get("resolved", eff.get("config", {}))
    sources = eff.get("sources", {})
    print("EFFECTIVE (resolved + per-field source):")
    for field in sorted(resolved.keys()):
        val = resolved[field]
        src = sources.get(field, "?")
        vs = json.dumps(val) if not isinstance(val, str) else val
        if len(str(vs)) > 70:
            vs = str(vs)[:67] + "..."
        marker = "*" if src == level else " "
        print(f"  {marker} {field:<30} {vs:<60} [{src}]")


In [ ]:
def put_config(plugin_id: str, body: dict, level: str = "collection") -> None:
    if level == "collection":
        url = f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/{plugin_id}"
    else:
        url = f"/configs/catalogs/{CATALOG_ID}/plugins/{plugin_id}"
    r = client.put(url, content=json.dumps(body), timeout=60.0)
    print(f"PUT {plugin_id}: {r.status_code}")
    if r.status_code not in (200, 201, 204):
        print(f"  body: {r.text[:300]}")
        if r.status_code == 401:
            raise RuntimeError("Unauthorized — set DYNASTORE_TOKEN.")
        raise RuntimeError(f"PUT failed: {r.status_code}")


## Step 1 — PATCH `items_postgresql_driver_config` with all 4 sidecars

Sidecars discriminated by `sidecar_type`.  The `geometries` sidecar's
`statistics` block defaults to area/length/centroid all `enabled=true,
index=true`; we set it explicitly here for documentation value.


In [ ]:
items_pg_driver = {
    "engine_ref": "postgresql_engine_config",
    "sidecars": [
        {
            "sidecar_type": "geometries",
            "target_srid": 4326,
            "target_dimension": "FORCE_2D",
            "geom_column": "geom",
            "bbox_column": "bbox_geom",
            "geohash_precision": 9,
            "store_bbox": True,
            "statistics": {
                "enabled": True,
                "storage_mode": "COLUMNAR",
                "area":     {"enabled": True, "index": True},
                "length":   {"enabled": True, "index": True},
                "centroid_type": "geometric",
                "index_centroid": True,
            },
        },
        {
            "sidecar_type": "attributes",
            "storage_mode": "AUTOMATIC",
            "enable_external_id": True,
            "external_id_field": "properties.code",
            "index_external_id": True,
            "enable_asset_id": True,
            "index_asset_id": True,
            "enable_validity": True,
        },
        {"sidecar_type": "item_metadata"},
        {"sidecar_type": "stac_metadata"},
    ],
}
put_config("items_postgresql_driver_config", items_pg_driver)
show_config_delta("items_postgresql_driver_config")


## Step 2 — PATCH `items_write_policy` for multi-version on `code`


In [ ]:
write_policy = {
    "on_conflict": "new_version",
    "identity_matchers": ["external_id"],
    "external_id_field": "properties.code",
    "track_asset_id": True,
    "enable_validity": True,
}
put_config("items_write_policy", write_policy)
show_config_delta("items_write_policy")


## Step 3 — PATCH `items_routing_config` for dual SEARCH dispatch

WRITE → PG (fatal, primary) + ES public (async, async outbox).  
SEARCH/READ → ES public with `geometry_simplified` (warn) + PG with
`geometry_exact` (fatal).  Caller chooses via `?hint=…`.


In [ ]:
routing = {
    "operations": {
        "WRITE": [
            {"driver_ref": "items_postgresql_driver",     "on_failure": "fatal"},
            {"driver_ref": "items_elasticsearch_driver",  "write_mode": "async", "on_failure": "outbox"},
        ],
        "READ": [
            {"driver_ref": "items_elasticsearch_driver",  "hints": ["geometry_simplified"], "on_failure": "warn"},
            {"driver_ref": "items_postgresql_driver",     "hints": ["geometry_exact"],     "on_failure": "fatal"},
        ],
        "SEARCH": [
            {"driver_ref": "items_elasticsearch_driver",  "hints": ["geometry_simplified"], "on_failure": "warn"},
            {"driver_ref": "items_postgresql_driver",     "hints": ["geometry_exact"],     "on_failure": "fatal"},
        ],
    },
}
put_config("items_routing_config", routing)
show_config_delta("items_routing_config")


## Step 4 — Confirm slim view shows everything

`strict=true&resolved=true` against the collection scope returns only
the configs owned at this scope (the four PUTs we just made).


In [ ]:
r = client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/",
    params={"strict": "true", "resolved": "true", "meta": "none"},
)
configs = r.json().get("configs", {})
items = configs.get("platform", {}).get("catalog", {}).get("collection", {}).get("items", {})
print("items.* keys:", list(items.keys()))
for sub, val in items.items():
    print(f"  {sub}:", list(val.keys()) if isinstance(val, dict) else type(val).__name__)


## Step 5 — Ingest sample features

Three features with distinct `code` values; routing fans out to PG
(synchronous) + ES (async, outbox-backed).


In [ ]:
sample_features = []
for i, code_val in enumerate(["AREA-001", "AREA-002", "AREA-003"]):
    sample_features.append({
        "type": "Feature",
        "stac_version": "1.0.0",
        "id": f"feat-{i}-{RUN_ID}",
        "collection": COLLECTION_ID,
        "geometry": {
            "type": "Polygon",
            "coordinates": [[
                [12.4 + i*0.1, 41.85], [12.55 + i*0.1, 41.85],
                [12.55 + i*0.1, 41.95], [12.4 + i*0.1, 41.95],
                [12.4 + i*0.1, 41.85],
            ]],
        },
        "bbox": [12.4 + i*0.1, 41.85, 12.55 + i*0.1, 41.95],
        "properties": {
            "datetime": "2024-01-10T00:00:00Z",
            "code": code_val,
            "name": f"Demo region {code_val}",
        },
        "assets": {},
        "links": [],
    })

ingest_url = f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items"
for feat in sample_features:
    r = client.post(ingest_url, content=json.dumps(feat))
    print(f"  POST {feat['id']}: {r.status_code}")
    assert r.status_code in (200, 201, 207), f"ingest failed: {r.status_code} {r.text[:200]}"


## Step 6 — Verify dual-search hint dispatch

Same query with two different hints — geometries returned by ES are
simplified (lower vertex count); PG returns exact stored polygons.


In [ ]:
search_url = f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items"
# Approximated path — ES with simplified geometry
r_es = client.get(search_url, params={"limit": 5, "hint": "geometry_simplified"})
print(f"SEARCH hint=geometry_simplified: HTTP {r_es.status_code}")
es_features = r_es.json().get("features", []) if r_es.status_code == 200 else []
print(f"  features returned: {len(es_features)}")

# Exact path — PG with stored polygon
r_pg = client.get(search_url, params={"limit": 5, "hint": "geometry_exact"})
print(f"SEARCH hint=geometry_exact: HTTP {r_pg.status_code}")
pg_features = r_pg.json().get("features", []) if r_pg.status_code == 200 else []
print(f"  features returned: {len(pg_features)}")

if es_features and pg_features:
    es_vertex_count = sum(len(c) for c in es_features[0]["geometry"]["coordinates"][0])
    pg_vertex_count = sum(len(c) for c in pg_features[0]["geometry"]["coordinates"][0])
    print(f"  vertex count: ES={es_vertex_count}, PG={pg_vertex_count}")
    print(f"  same? {es_vertex_count == pg_vertex_count} — equal is fine for tiny geoms,"
          f" different proves hint dispatch")


## Teardown

In [ ]:
# Teardown — delete the ephemeral catalog. Comment out to keep state.
r = client.delete(
    f"/stac/catalogs/{CATALOG_ID}",
    params={"force": "true"},
    timeout=60.0,
)
print(f"teardown DELETE /stac/catalogs/{CATALOG_ID}: {r.status_code}")
client.close()
